# Motif frequencies

This notebook loads the tutorial 02 serialized CISM artifact and summarizes motif frequencies. It includes a single-patient motif-frequency histogram, NN/NP average frequencies across FOVs, and motif-level relative enrichment columns.

## Setup

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except ImportError:
    sns = None

if sns is not None:
    sns.set_theme(style="whitegrid", context="notebook")
else:
    plt.style.use("ggplot")


def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "cism").exists() and (candidate / "melanoma_tutorials").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current notebook path.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

TUTORIAL_DIR = REPO_ROOT / "melanoma_tutorials"
RUNTIME_DIR = TUTORIAL_DIR / "melanoma_tutorials" / "runtime"
ARTIFACT_PATH = RUNTIME_DIR / "serialized" / "Melanoma_cism_ready.pkl"

ARTIFACT_PATH

## Load tutorial 02 artifact

In [ ]:
with ARTIFACT_PATH.open("rb") as handle:
    artifact = pickle.load(handle)

cism = artifact["cism"]
motif_df = cism.motifs_dataset.copy()

print(f"Loaded artifact: {ARTIFACT_PATH}")
print(f"Motif rows: {motif_df.shape[0]:,}")
print(f"Unique motifs: {motif_df['ID'].nunique():,}")
print(f"Patient classes: {sorted(motif_df['patient_class'].dropna().unique())}")

display(motif_df.head())

## Single Patient Motif-Frequency Histogram

Choose one patient and plot how many motifs fall in each patient-level frequency range. This helps inspect the high-frequency tail.

In [ ]:
available_patients = (
    motif_df[["Patient_uId", "patient_class"]]
    .drop_duplicates()
    .sort_values(["patient_class", "Patient_uId"])
    .reset_index(drop=True)
)

patient_fov_counts = (
    motif_df[["Patient_uId", "FOV"]]
    .drop_duplicates()
    .groupby("Patient_uId", observed=True)["FOV"]
    .nunique()
    .rename("n_fovs")
)
available_patients = available_patients.join(patient_fov_counts, on="Patient_uId")

display(available_patients.head(20))

# Edit this value to inspect another patient.
SELECTED_PATIENT_UID = available_patients.iloc[0]["Patient_uId"]
FREQUENCY_BINS = np.linspace(0, 0.5, 101)
USE_LOG_Y_AXIS = True

selected_patient_df = motif_df.loc[motif_df["Patient_uId"] == SELECTED_PATIENT_UID].copy()
if selected_patient_df.empty:
    raise ValueError(f"No motif rows found for patient {SELECTED_PATIENT_UID!r}")

selected_patient_class = selected_patient_df["patient_class"].iloc[0]
selected_patient_fov_count = selected_patient_df["FOV"].nunique()
patient_motif_frequency = (
    selected_patient_df.groupby("ID", observed=True)["Freq"]
    .sum()
    .div(selected_patient_fov_count)
    .rename("mean_frequency_across_patient_fovs")
)

print(
    f"Selected {SELECTED_PATIENT_UID}: "
    f"{patient_motif_frequency.shape[0]:,} motifs across "
    f"{selected_patient_fov_count:,} FOVs, class={selected_patient_class}"
)

fig, ax = plt.subplots(figsize=(10, 6))
counts, bins, patches = ax.hist(
    patient_motif_frequency,
    bins=FREQUENCY_BINS,
    color="#4C78A8",
    edgecolor="white",
    linewidth=0.4,
)

if USE_LOG_Y_AXIS:
    ax.set_yscale("log")

ax.set_xlim(FREQUENCY_BINS[0], 0.1)
ax.set_title(f"Patient-level motif frequency distribution: {SELECTED_PATIENT_UID}")
ax.set_xlabel("Mean motif frequency across selected patient's FOVs (missing FOVs as 0)")
ax.set_ylabel("Number of motifs in frequency bin")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

histogram_table = pd.DataFrame(
    {
        "bin_left": bins[:-1],
        "bin_right": bins[1:],
        "motif_count": counts.astype(int),
    }
)
display(histogram_table.loc[histogram_table["motif_count"] > 0].tail(10))

## Average Frequency And Relative Enrichment Across NN/NP FOVs

A fold here is one patient/FOV sample. The motif table only contains observed motifs, so missing motif/FOV combinations are treated as `0` before computing each class mean.

The enrichment columns compare the NN and NP average frequencies for each motif.

A motif must also pass the majority-class patient-prevalence stringency rule: it is kept only if it appears in at least 30% of patients from the class where its mean frequency is higher.

In [ ]:
LABELS_TO_KEEP = ["NN", "NP"]
TOP_N_MOTIFS = 30
#ENRICHMENT_EPSILON = 1e-12
MAJORITY_CLASS_PATIENT_PREVALENCE_MIN = 0.30
MEAN_FREQUENCY_COLUMNS = {
    label: f"{label}_mean_frequency"
    for label in LABELS_TO_KEEP
}
PREVALENCE_COLUMNS = {
    label: f"{label}_patient_prevalence"
    for label in LABELS_TO_KEEP
}

required_columns = {"ID", "Freq", "Patient_uId", "FOV", "patient_class"}
missing_columns = required_columns - set(motif_df.columns)
if missing_columns:
    raise ValueError(f"Missing required motif table columns: {sorted(missing_columns)}")

filtered = motif_df.loc[motif_df["patient_class"].isin(LABELS_TO_KEEP)].copy()
filtered["fold_id"] = filtered["Patient_uId"].astype(str) + "_" + filtered["FOV"].astype(str)
ENRICHMENT_EPSILON = filtered['Freq'].min()

fold_meta = (
    filtered[["fold_id", "patient_class"]]
    .drop_duplicates()
    .sort_values(["patient_class", "fold_id"])
    .reset_index(drop=True)
)

class_fold_count = fold_meta.groupby("patient_class", observed=True)["fold_id"].nunique()
patient_meta = (
    filtered[["Patient_uId", "patient_class"]]
    .drop_duplicates()
    .sort_values(["patient_class", "Patient_uId"])
    .reset_index(drop=True)
)
class_patient_count = patient_meta.groupby("patient_class", observed=True)["Patient_uId"].nunique()

frequency_summary = (
    filtered.groupby(["ID", "patient_class"], observed=True)["Freq"]
    .sum()
    .unstack("patient_class", fill_value=0.0)
    .reindex(columns=LABELS_TO_KEEP, fill_value=0.0)
)

for label in LABELS_TO_KEEP:
    frequency_summary[label] = frequency_summary[label] / class_fold_count[label]

frequency_summary = frequency_summary.rename(columns=MEAN_FREQUENCY_COLUMNS)
nn_col = MEAN_FREQUENCY_COLUMNS["NN"]
np_col = MEAN_FREQUENCY_COLUMNS["NP"]
mean_frequency_cols = [nn_col, np_col]

frequency_summary["total_mean_frequency"] = frequency_summary[mean_frequency_cols].mean(axis=1)
frequency_summary["frequency_difference_nn_minus_np"] = frequency_summary[nn_col] - frequency_summary[np_col]
frequency_summary["abs_difference"] = frequency_summary["frequency_difference_nn_minus_np"].abs()
frequency_summary["frequency_ratio_nn_over_np"] = (
    frequency_summary[nn_col] + ENRICHMENT_EPSILON
) / (frequency_summary[np_col] + ENRICHMENT_EPSILON)
frequency_summary["log_ratio_nn_over_np"] = np.log(frequency_summary["frequency_ratio_nn_over_np"])
frequency_summary["abs_log_ratio"] = frequency_summary["log_ratio_nn_over_np"].abs()
frequency_summary["dominant_label"] = frequency_summary[mean_frequency_cols].idxmax(axis=1).str.replace(
    "_mean_frequency", "", regex=False
)

patient_prevalence = (
    filtered.loc[filtered["Freq"] > 0, ["ID", "Patient_uId", "patient_class"]]
    .drop_duplicates()
    .groupby(["ID", "patient_class"], observed=True)["Patient_uId"]
    .nunique()
    .unstack("patient_class", fill_value=0.0)
    .reindex(index=frequency_summary.index, columns=LABELS_TO_KEEP, fill_value=0.0)
)
for label in LABELS_TO_KEEP:
    patient_prevalence[label] = patient_prevalence[label] / class_patient_count[label]
patient_prevalence = patient_prevalence.rename(columns=PREVALENCE_COLUMNS)
frequency_summary = frequency_summary.join(patient_prevalence)

frequency_summary["majority_class_patient_prevalence"] = np.select(
    [frequency_summary["dominant_label"] == label for label in LABELS_TO_KEEP],
    [frequency_summary[PREVALENCE_COLUMNS[label]] for label in LABELS_TO_KEEP],
    default=np.nan,
)
frequency_summary["passes_majority_prevalence_stringency"] = (
    frequency_summary["majority_class_patient_prevalence"] >= MAJORITY_CLASS_PATIENT_PREVALENCE_MIN
)
frequency_summary = frequency_summary.sort_values("total_mean_frequency", ascending=False)
stringent_frequency_summary = frequency_summary.loc[
    frequency_summary["passes_majority_prevalence_stringency"]
].copy()

print(f"NN/NP FOVs: {len(fold_meta):,}")
print(f"NN/NP patients: {len(patient_meta):,}")
print(
    f"Motifs passing majority-class patient prevalence >= "
    f"{MAJORITY_CLASS_PATIENT_PREVALENCE_MIN:.0%}: "
    f"{len(stringent_frequency_summary):,} / {len(frequency_summary):,}"
)
display(fold_meta["patient_class"].value_counts().rename("fov_count"))
display(patient_meta["patient_class"].value_counts().rename("patient_count"))
display(stringent_frequency_summary.head(TOP_N_MOTIFS))

## Motifs With Strongest Absolute NN/NP Difference

In [ ]:
display(
    stringent_frequency_summary
    .sort_values("abs_difference", ascending=False)
    .head(TOP_N_MOTIFS)
)

## Motifs With Strongest Absolute Log Ratio

In [ ]:
display(
    stringent_frequency_summary
    .sort_values("abs_log_ratio", ascending=False)
    .head(TOP_N_MOTIFS)
)

## Top Motif Enrichment Plots

These plots show the motifs with the strongest NN/NP separation by absolute frequency difference and by `abs(log(NN / NP))`. Bars are blue when the motif is enriched toward NN and orange when enriched toward NP.

In [ ]:
DIRECTION_COLORS = {"NN": "#4C78A8", "NP": "#F58518"}


def plot_directional_metric(metric_col, title, xlabel, top_n=TOP_N_MOTIFS):
    plot_df = (
        stringent_frequency_summary
        .sort_values(metric_col, ascending=False)
        .head(top_n)
        .sort_values(metric_col)
        .copy()
    )
    motif_labels = plot_df.index.astype(str)
    y = np.arange(len(plot_df))
    direction_colors = plot_df["dominant_label"].map(DIRECTION_COLORS).fillna("#9E9E9E")

    fig_height = max(6, 0.32 * len(plot_df))
    fig, (ax_metric, ax_frequency) = plt.subplots(
        ncols=2,
        sharey=True,
        figsize=(14, fig_height),
        gridspec_kw={"width_ratios": [1.05, 1.0], "wspace": 0.08},
    )

    ax_metric.barh(
        y,
        plot_df[metric_col],
        color=direction_colors,
        edgecolor="white",
        linewidth=0.5,
    )
    ax_metric.set_title(title)
    ax_metric.set_xlabel(xlabel)
    ax_metric.set_ylabel("Motif ID")
    ax_metric.set_yticks(y)
    ax_metric.set_yticklabels(motif_labels)
    ax_metric.grid(axis="x", alpha=0.25)
    ax_metric.grid(axis="y", visible=False)

    bar_height = 0.38
    ax_frequency.barh(
        y - bar_height / 2,
        plot_df[MEAN_FREQUENCY_COLUMNS["NN"]],
        height=bar_height,
        color=DIRECTION_COLORS["NN"],
        edgecolor="white",
        linewidth=0.5,
        label="NN",
    )
    ax_frequency.barh(
        y + bar_height / 2,
        plot_df[MEAN_FREQUENCY_COLUMNS["NP"]],
        height=bar_height,
        color=DIRECTION_COLORS["NP"],
        edgecolor="white",
        linewidth=0.5,
        label="NP",
    )
    ax_frequency.set_title("Class mean frequency")
    ax_frequency.set_xlabel("Average motif frequency across FOVs")
    ax_frequency.tick_params(axis="y", left=False, labelleft=False)
    ax_frequency.grid(axis="x", alpha=0.25)
    ax_frequency.grid(axis="y", visible=False)
    ax_frequency.legend(title="Patient class")

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=DIRECTION_COLORS[label])
        for label in LABELS_TO_KEEP
    ]
    ax_metric.legend(handles, LABELS_TO_KEEP, title="Enriched toward")
    plt.tight_layout()
    plt.show()


plot_directional_metric(
    "abs_difference",
    "Top motifs by absolute NN/NP frequency difference",
    "abs(NN mean frequency - NP mean frequency)",
)

plot_directional_metric(
    "abs_log_ratio",
    "Top motifs by absolute log frequency ratio",
    "abs(log(NN mean frequency / NP mean frequency))",
)


## NN/NP Split Histogram

This binned histogram shows the distribution of motif average frequencies. Each bin is stacked by whether the class-specific average came from `NN` or `NP` FOVs.

In [ ]:
hist_values = [stringent_frequency_summary[MEAN_FREQUENCY_COLUMNS[label]].to_numpy() for label in LABELS_TO_KEEP]

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(
    hist_values,
    bins=50,
    stacked=True,
    label=LABELS_TO_KEEP,
    color=["#4C78A8", "#F58518"],
    edgecolor="white",
    linewidth=0.4,
)
ax.set_title("Distribution of motif average frequency across NN/NP FOVs")
ax.set_xlabel("Average motif frequency across FOVs")
ax.set_ylabel("Motif count")
ax.legend(title="Patient class")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Top Motif Split Bars

This view names the top motifs and splits each motif bar into its `NN` and `NP` average-frequency components.

In [ ]:
plot_df = stringent_frequency_summary.head(TOP_N_MOTIFS).sort_values("total_mean_frequency")

fig, ax = plt.subplots(figsize=(10, max(6, 0.28 * len(plot_df))))
left = np.zeros(len(plot_df))
colors = {"NN": "#4C78A8", "NP": "#F58518"}

for label in LABELS_TO_KEEP:
    ax.barh(
        plot_df.index.astype(str),
        plot_df[MEAN_FREQUENCY_COLUMNS[label]],
        left=left,
        color=colors[label],
        label=label,
        edgecolor="white",
        linewidth=0.5,
    )
    left += plot_df[MEAN_FREQUENCY_COLUMNS[label]].to_numpy()

ax.set_title("Motif average frequency across FOVs: NN and NP")
ax.set_xlabel("Average motif frequency across FOVs")
ax.set_ylabel("Motif ID")
ax.legend(title="Patient class")
ax.grid(axis="x", alpha=0.25)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## Random Forest LOOCV With Optuna-Tuned Stringency

This section runs two leakage-safe Random Forest LOOCV analyses. In each held-out patient fold, motifs are re-ranked using only the training patients, with at most 30 selected features.

Optuna tunes the majority-class patient-prevalence stringency separately for the `abs_difference` and `abs_log_ratio` rankings.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, roc_auc_score

from cism.motif_selection import build_patient_motif_matrix

try:
    import optuna
except ImportError as exc:
    raise ImportError("Optuna is required for this section. Install it with `pip install optuna`.") from exc

RF_RANDOM_STATE = 0
OPTUNA_STRINGENCY_TRIALS = 30
STRINGENCY_STEP = 0.01
RF_TOP_K = min(30, TOP_N_MOTIFS)
RF_POSITIVE_LABEL = "NN"
RF_NEGATIVE_LABEL = "NP"
RF_LABEL_ORDER = [RF_POSITIVE_LABEL, RF_NEGATIVE_LABEL]

rf_patient_classes = (
    filtered[["Patient_uId", "patient_class"]]
    .drop_duplicates()
    .set_index("Patient_uId")
    .loc[:, "patient_class"]
    .sort_index()
)
rf_patient_classes = rf_patient_classes.loc[rf_patient_classes.isin(RF_LABEL_ORDER)]

rf_motif_df = filtered.loc[filtered["Patient_uId"].isin(rf_patient_classes.index)].copy()
rf_patient_motif_matrix = build_patient_motif_matrix(
    rf_motif_df,
    patient_ids=rf_patient_classes.index,
)

rf_fov_frequency = (
    rf_motif_df.groupby(["ID", "Patient_uId", "FOV", "patient_class"], observed=True)["Freq"]
    .sum()
    .reset_index()
)
rf_patient_presence = (
    rf_motif_df.loc[rf_motif_df["Freq"] > 0, ["ID", "Patient_uId", "patient_class"]]
    .drop_duplicates()
)

print(f"RF patients: {rf_patient_classes.shape[0]:,}")
print(f"RF candidate motifs: {rf_patient_motif_matrix.shape[1]:,}")
display(rf_patient_classes.value_counts().rename("patient_count"))

In [ ]:
def score_motifs_for_patients(patient_ids, majority_prevalence_min):
    patient_ids = pd.Index(patient_ids)
    local_fov_frequency = rf_fov_frequency.loc[rf_fov_frequency["Patient_uId"].isin(patient_ids)].copy()
    local_presence = rf_patient_presence.loc[rf_patient_presence["Patient_uId"].isin(patient_ids)].copy()

    fold_meta_local = local_fov_frequency[["Patient_uId", "FOV", "patient_class"]].drop_duplicates()
    class_fold_count_local = fold_meta_local.groupby("patient_class", observed=True)["FOV"].count()

    patient_meta_local = (
        rf_patient_classes.loc[patient_ids]
        .rename("patient_class")
        .reset_index()
        .drop_duplicates()
    )
    class_patient_count_local = patient_meta_local.groupby("patient_class", observed=True)["Patient_uId"].nunique()

    summary = (
        local_fov_frequency.groupby(["ID", "patient_class"], observed=True)["Freq"]
        .sum()
        .unstack("patient_class", fill_value=0.0)
        .reindex(columns=RF_LABEL_ORDER, fill_value=0.0)
    )
    for label in RF_LABEL_ORDER:
        summary[label] = summary[label] / class_fold_count_local[label]

    summary = summary.rename(columns=MEAN_FREQUENCY_COLUMNS)
    nn_col = MEAN_FREQUENCY_COLUMNS["NN"]
    np_col = MEAN_FREQUENCY_COLUMNS["NP"]
    mean_frequency_cols = [nn_col, np_col]

    summary["total_mean_frequency"] = summary[mean_frequency_cols].mean(axis=1)
    summary["frequency_difference_nn_minus_np"] = summary[nn_col] - summary[np_col]
    summary["abs_difference"] = summary["frequency_difference_nn_minus_np"].abs()
    summary["frequency_ratio_nn_over_np"] = (
        summary[nn_col] + ENRICHMENT_EPSILON
    ) / (summary[np_col] + ENRICHMENT_EPSILON)
    summary["log_ratio_nn_over_np"] = np.log(summary["frequency_ratio_nn_over_np"])
    summary["abs_log_ratio"] = summary["log_ratio_nn_over_np"].abs()
    summary["dominant_label"] = summary[mean_frequency_cols].idxmax(axis=1).str.replace(
        "_mean_frequency", "", regex=False
    )

    prevalence = (
        local_presence.groupby(["ID", "patient_class"], observed=True)["Patient_uId"]
        .nunique()
        .unstack("patient_class", fill_value=0.0)
        .reindex(index=summary.index, columns=RF_LABEL_ORDER, fill_value=0.0)
    )
    for label in RF_LABEL_ORDER:
        prevalence[label] = prevalence[label] / class_patient_count_local[label]
    prevalence = prevalence.rename(columns=PREVALENCE_COLUMNS)
    summary = summary.join(prevalence)

    summary["majority_class_patient_prevalence"] = np.select(
        [summary["dominant_label"] == label for label in RF_LABEL_ORDER],
        [summary[PREVALENCE_COLUMNS[label]] for label in RF_LABEL_ORDER],
        default=np.nan,
    )
    summary["passes_majority_prevalence_stringency"] = (
        summary["majority_class_patient_prevalence"] >= majority_prevalence_min
    )
    return summary


def select_top_motifs_for_patients(patient_ids, metric_col, majority_prevalence_min, top_k=RF_TOP_K):
    score_table = score_motifs_for_patients(patient_ids, majority_prevalence_min)
    selected = (
        score_table.loc[score_table["passes_majority_prevalence_stringency"]]
        .sort_values(metric_col, ascending=False)
        .head(top_k)
        .index
        .tolist()
    )
    return selected, score_table


def summarize_rf_loocv_results(results_df):
    y_true = results_df["class"]
    y_pred = results_df["pred_class"]
    y_binary = (y_true == RF_POSITIVE_LABEL).astype(int)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mean_selected_motifs": float(results_df["selected_feature_count"].mean()),
        "median_selected_motifs": float(results_df["selected_feature_count"].median()),
        "valid_folds": int(results_df.shape[0]),
    }
    metrics["roc_auc"] = (
        float(roc_auc_score(y_binary, results_df["positive_probability"]))
        if y_binary.nunique() == 2
        else np.nan
    )
    return metrics


def evaluate_rf_loocv(metric_col, majority_prevalence_min, random_state=RF_RANDOM_STATE, top_k=RF_TOP_K):
    rows = []
    selected_by_patient = {}
    score_tables_by_patient = {}

    for held_out_patient in rf_patient_classes.index:
        train_patients = rf_patient_classes.index.difference([held_out_patient], sort=False)
        selected, score_table = select_top_motifs_for_patients(
            train_patients,
            metric_col=metric_col,
            majority_prevalence_min=majority_prevalence_min,
            top_k=top_k,
        )
        selected_by_patient[str(held_out_patient)] = selected
        score_tables_by_patient[str(held_out_patient)] = score_table
        if not selected:
            raise ValueError(
                f"No motifs selected for held-out patient {held_out_patient} "
                f"with {metric_col=} and {majority_prevalence_min=:.2f}."
            )

        x_train = rf_patient_motif_matrix.loc[train_patients].reindex(columns=selected, fill_value=0.0)
        y_train = rf_patient_classes.loc[train_patients]
        x_test = rf_patient_motif_matrix.loc[[held_out_patient]].reindex(columns=selected, fill_value=0.0)
        y_test = rf_patient_classes.loc[held_out_patient]

        clf = RandomForestClassifier(random_state=random_state, n_jobs=-1)
        clf.fit(x_train, y_train)
        pred = clf.predict(x_test)[0]
        probabilities = clf.predict_proba(x_test)[0]
        positive_index = int(np.where(clf.classes_ == RF_POSITIVE_LABEL)[0][0])
        positive_probability = float(probabilities[positive_index])

        rows.append(
            {
                "patient_id": held_out_patient,
                "class": y_test,
                "pred_class": pred,
                "positive_probability": positive_probability,
                "selected_feature_count": len(selected),
                "selected_motif_ids": selected,
                "majority_prevalence_min": majority_prevalence_min,
                "metric_col": metric_col,
            }
        )

    results_df = pd.DataFrame(rows).set_index("patient_id")
    metrics = summarize_rf_loocv_results(results_df)
    return {
        "metric_col": metric_col,
        "majority_prevalence_min": majority_prevalence_min,
        "results": results_df,
        "metrics": metrics,
        "selected_by_patient": selected_by_patient,
        "score_tables_by_patient": score_tables_by_patient,
    }


def optimize_stringency_for_rf_metric(metric_col, n_trials=OPTUNA_STRINGENCY_TRIALS):
    def objective(trial):
        majority_prevalence_min = trial.suggest_float(
            "majority_prevalence_min",
            0.0,
            1.0,
            step=STRINGENCY_STEP,
        )
        try:
            evaluation = evaluate_rf_loocv(
                metric_col=metric_col,
                majority_prevalence_min=majority_prevalence_min,
            )
        except ValueError:
            trial.set_user_attr("valid_folds", 0)
            trial.set_user_attr("mean_selected_motifs", 0.0)
            return 0.0

        metrics = evaluation["metrics"]
        trial.set_user_attr("roc_auc", metrics["roc_auc"])
        trial.set_user_attr("accuracy", metrics["accuracy"])
        trial.set_user_attr("balanced_accuracy", metrics["balanced_accuracy"])
        trial.set_user_attr("mean_selected_motifs", metrics["mean_selected_motifs"])
        trial.set_user_attr("valid_folds", metrics["valid_folds"])
        return metrics["roc_auc"] if np.isfinite(metrics["roc_auc"]) else 0.0

    sampler = optuna.samplers.TPESampler(seed=RF_RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_threshold = float(study.best_params["majority_prevalence_min"])
    best_evaluation = evaluate_rf_loocv(metric_col, best_threshold)
    trials_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
    return study, trials_df, best_evaluation

In [ ]:
rf_optimization_outputs = {}
rf_best_rows = []

for metric_col, analysis_name in [
    ("abs_difference", "absolute_difference"),
    ("abs_log_ratio", "absolute_log_ratio"),
]:
    print(f"Optimizing stringency for {analysis_name}...")
    study, trials_df, evaluation = optimize_stringency_for_rf_metric(metric_col)
    rf_optimization_outputs[analysis_name] = {
        "study": study,
        "trials": trials_df,
        "evaluation": evaluation,
    }
    rf_best_rows.append(
        {
            "analysis": analysis_name,
            "metric_col": metric_col,
            "best_majority_prevalence_min": evaluation["majority_prevalence_min"],
            **evaluation["metrics"],
        }
    )

rf_best_metrics_df = pd.DataFrame(rf_best_rows).sort_values("roc_auc", ascending=False)
display(rf_best_metrics_df)

for analysis_name, output in rf_optimization_outputs.items():
    print(f"{analysis_name} Optuna trials")
    display(output["trials"].sort_values("value", ascending=False).head(10))
    print(f"{analysis_name} LOOCV results")
    display(output["evaluation"]["results"].drop(columns=["selected_motif_ids"]).head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
metric_plot_df = rf_best_metrics_df.melt(
    id_vars="analysis",
    value_vars=["roc_auc", "accuracy", "balanced_accuracy"],
    var_name="metric",
    value_name="value",
)
if sns is not None:
    sns.barplot(data=metric_plot_df, x="analysis", y="value", hue="metric", ax=ax)
else:
    for metric_name, sub_df in metric_plot_df.groupby("metric"):
        ax.bar(sub_df["analysis"], sub_df["value"], alpha=0.7, label=metric_name)
ax.set_ylim(0, 1)
ax.set_title("RF LOOCV performance after Optuna stringency tuning")
ax.set_xlabel("Analysis")
ax.set_ylabel("Score")
ax.legend(title="Metric")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(rf_optimization_outputs), figsize=(5 * len(rf_optimization_outputs), 4))
if len(rf_optimization_outputs) == 1:
    axes = [axes]

for ax, (analysis_name, output) in zip(axes, rf_optimization_outputs.items()):
    results = output["evaluation"]["results"]
    matrix = confusion_matrix(results["class"], results["pred_class"], labels=RF_LABEL_ORDER)
    confusion_df = pd.DataFrame(matrix, index=RF_LABEL_ORDER, columns=RF_LABEL_ORDER)
    if sns is not None:
        sns.heatmap(confusion_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    else:
        image = ax.imshow(confusion_df.to_numpy(dtype=float), cmap="Blues")
        for row in range(confusion_df.shape[0]):
            for col in range(confusion_df.shape[1]):
                ax.text(col, row, str(confusion_df.iloc[row, col]), ha="center", va="center")
    ax.set_title(f"{analysis_name} confusion matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(np.arange(len(RF_LABEL_ORDER)))
    ax.set_xticklabels(RF_LABEL_ORDER)
    ax.set_yticks(np.arange(len(RF_LABEL_ORDER)))
    ax.set_yticklabels(RF_LABEL_ORDER)

plt.tight_layout()
plt.show()

## Export Table For Reuse

In [ ]:
OUTPUT_DIR = RUNTIME_DIR / "motif_frequencies"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = OUTPUT_DIR / "nn_np_motif_average_frequency_across_fovs_with_enrichment.csv"
stringent_frequency_summary.reset_index().to_csv(summary_path, index=False)

summary_path